# Example Gradio Pandas App

This notebook demonstrates how to create a Gradio app in JupyterLab that others can access on port 7860.

In [ ]:
import gradio as gr
import pandas as pd
import plotly.express as px

## Simple CSV Analyzer

Upload a CSV file and get a statistical summary.

In [ ]:
def analyze_csv(file):
    """Analyze uploaded CSV file"""
    if file is None:
        return "Please upload a CSV file"
    
    # Read the CSV file
    df = pd.read_csv(file.name)
    
    # Generate summary statistics
    summary = df.describe().to_html()
    info = f"""
    <h2>Dataset Overview</h2>
    <p><strong>Rows:</strong> {len(df)}</p>
    <p><strong>Columns:</strong> {len(df.columns)}</p>
    <p><strong>Column Names:</strong> {', '.join(df.columns)}</p>
    <h3>Statistical Summary</h3>
    {summary}
    """
    
    return info

# Create the Gradio interface
demo = gr.Interface(
    fn=analyze_csv,
    inputs=gr.File(label="Upload CSV File"),
    outputs=gr.HTML(label="Analysis Results"),
    title="📊 CSV Data Analyzer",
    description="Upload a CSV file to see its statistical summary",
    theme="soft"
)

# Launch the app on port 7860 (accessible to others)
# The app will be available at: https://your-workbench-url:7860
demo.launch(
    server_name="0.0.0.0",  # Allow external access
    server_port=7860,        # Use port 7860 (exposed in docker-compose)
    share=False              # No need for Gradio share in Workbench
)

## Interactive Data Dashboard

A more advanced example with multiple tabs and interactive visualizations.

In [ ]:
def create_plot(csv_file, x_col, y_col, chart_type):
    """Create interactive plots from CSV data"""
    if csv_file is None:
        return None
    
    df = pd.read_csv(csv_file.name)
    
    if chart_type == "Scatter":
        fig = px.scatter(df, x=x_col, y=y_col, title=f"{y_col} vs {x_col}")
    elif chart_type == "Line":
        fig = px.line(df, x=x_col, y=y_col, title=f"{y_col} over {x_col}")
    elif chart_type == "Bar":
        fig = px.bar(df, x=x_col, y=y_col, title=f"{y_col} by {x_col}")
    elif chart_type == "Histogram":
        fig = px.histogram(df, x=x_col, title=f"Distribution of {x_col}")
    
    return fig

def get_columns(file):
    """Extract column names from uploaded CSV"""
    if file is None:
        return gr.Dropdown(choices=[]), gr.Dropdown(choices=[])
    
    df = pd.read_csv(file.name)
    cols = df.columns.tolist()
    
    return (
        gr.Dropdown(choices=cols, value=cols[0] if cols else None),
        gr.Dropdown(choices=cols, value=cols[1] if len(cols) > 1 else cols[0] if cols else None)
    )

def show_data_preview(file):
    """Show first few rows of the data"""
    if file is None:
        return "Upload a file to see preview"
    
    df = pd.read_csv(file.name)
    return df.head(10).to_html(index=False)

# Create a multi-tab Gradio app
with gr.Blocks(theme="soft", title="Data Dashboard") as dashboard:
    gr.Markdown("# 📈 Interactive Data Dashboard")
    gr.Markdown("Upload a CSV file and explore your data with interactive visualizations")
    
    # Shared file upload
    file_input = gr.File(label="📁 Upload CSV File")
    
    with gr.Tabs():
        # Tab 1: Data Preview
        with gr.Tab("Data Preview"):
            preview_output = gr.HTML(label="First 10 Rows")
            file_input.change(
                fn=show_data_preview,
                inputs=file_input,
                outputs=preview_output
            )
        
        # Tab 2: Statistics
        with gr.Tab("Statistics"):
            stats_output = gr.HTML(label="Statistical Summary")
            file_input.change(
                fn=analyze_csv,
                inputs=file_input,
                outputs=stats_output
            )
        
        # Tab 3: Visualization
        with gr.Tab("Visualization"):
            with gr.Row():
                with gr.Column():
                    chart_type = gr.Radio(
                        choices=["Scatter", "Line", "Bar", "Histogram"],
                        label="Chart Type",
                        value="Scatter"
                    )
                    x_axis = gr.Dropdown(label="X Axis", choices=[])
                    y_axis = gr.Dropdown(label="Y Axis", choices=[])
                
                with gr.Column():
                    plot_output = gr.Plot(label="Visualization")
            
            # Update dropdowns when file is uploaded
            file_input.change(
                fn=get_columns,
                inputs=file_input,
                outputs=[x_axis, y_axis]
            )
            
            # Update plot when inputs change
            for component in [x_axis, y_axis, chart_type]:
                component.change(
                    fn=create_plot,
                    inputs=[file_input, x_axis, y_axis, chart_type],
                    outputs=plot_output
                )

# Launch the dashboard on port 7860
dashboard.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=False
)

## How to Use

1. Run one of the cells above to start the Gradio app
2. The app will be accessible on **port 7860**
3. Share the URL with others: `https://your-workbench-url:7860`
4. To stop the app, interrupt the cell or run: `demo.close()` or `dashboard.close()`

## Tips

- Only one Gradio app can run on port 7860 at a time
- To run multiple apps, close the current one first
- You can also combine multiple interfaces using tabs (as shown in the dashboard example)
- All changes to the code require restarting the app